# T-LARS Iterative Image Reconstruction

**Live sparse recovery with a DCT dictionary — watch the image emerge coefficient by coefficient.**

This notebook demonstrates the **Tensor Least Angle Regression and Selection (T-LARS)** algorithm
from the `tensor_ml` library applied to 2-D image reconstruction.  A standard **Discrete Cosine
Transform (DCT)** dictionary is used as the sparsifying basis.

### What you will see

| Step | Description |
|------|-------------|
| **1** | Build a small grayscale test image and an overcomplete DCT dictionary |
| **2** | Run T-LARS with `debug_mode=True` so every iteration prints to the terminal |
| **3** | After each checkpoint the **reconstructed image** is displayed inline |
| **4** | Convergence curves, coefficient paths, and final quality metrics |
| **5** | Edge-case and advanced-usage examples |

> **Requirements:** `numpy`, `matplotlib`, `tensor_ml` (this repo).  
> PyTorch is optional — all examples use the NumPy backend.

## 1. Install and Import Dependencies

In [ ]:
"""
Install and Import Dependencies
================================
tensor_ml must be installed from the repo root (``pip install -e .``).
numpy and matplotlib are the only runtime requirements for this notebook.
"""

# Uncomment the lines below if running for the first time:
# %pip install -e ../..     # install tensor_ml from repo root
# %pip install matplotlib

import logging
import sys
import time
from typing import List, Tuple

import numpy as np
import matplotlib.pyplot as plt
from IPython import display as ipy_display

from tensor_ml import TLARS
from tensor_ml.tensor_models.multilinear.tlars import TLARSConfig

print("All imports successful ✓")

## 2. Configure Logging for Line-by-Line Terminal Output

T-LARS uses Python's standard `logging` module.  When you set `debug_mode=True`
in the TLARS constructor **and** configure the root logger to `DEBUG`, every
iteration prints:

* iteration number, active-set size, current λ
* columns entering / leaving the active set
* stopping-condition triggers

The cell below routes those messages to **stdout** so they appear in the
notebook output (or your terminal if running as a script).

In [ ]:
# ---------------------------------------------------------------------------
# Logging configuration
# ---------------------------------------------------------------------------
# Remove any existing handlers to avoid duplicates when re-running the cell.
tlars_logger = logging.getLogger("tensor_ml.tensor_models.multilinear.tlars")
tlars_logger.handlers.clear()

handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.DEBUG)
formatter = logging.Formatter(
    "%(asctime)s | %(levelname)-5s | %(message)s",
    datefmt="%H:%M:%S",
)
handler.setFormatter(formatter)
tlars_logger.addHandler(handler)
tlars_logger.setLevel(logging.DEBUG)

print("Logger configured — DEBUG messages will appear below each TLARS run.")

## 3. Load and Preprocess a Sample Image

We generate a **synthetic 32 × 32 grayscale** test image containing simple
geometric shapes.  This keeps the demo self-contained (no external files) and
small enough that T-LARS converges quickly.

> You can replace this with any grayscale image by loading it with
> `PIL.Image.open(...)` and resizing to `(N, N)`.

In [ ]:
def make_test_image(size: int = 32) -> np.ndarray:
    """Generate a synthetic grayscale test image with geometric shapes.

    Parameters
    ----------
    size : int, default=32
        Height and width of the square output image.

    Returns
    -------
    image : np.ndarray, shape (size, size)
        Pixel values in [0, 1].

    Examples
    --------
    >>> img = make_test_image(32)
    >>> img.shape
    (32, 32)
    """
    img = np.zeros((size, size), dtype=np.float64)

    # Central bright rectangle
    s4, s3_4 = size // 4, 3 * size // 4
    img[s4:s3_4, s4:s3_4] = 0.9

    # Smaller dark square inside
    s3_8, s5_8 = 3 * size // 8, 5 * size // 8
    img[s3_8:s5_8, s3_8:s5_8] = 0.3

    # Two bright dots (top-left and bottom-right)
    img[2, 2] = 1.0
    img[-3, -3] = 1.0

    # Horizontal gradient stripe across the top
    img[1, :] = np.linspace(0, 1, size)

    return img


# Generate and display
Y_image = make_test_image(32)

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(Y_image, cmap="gray", vmin=0, vmax=1)
ax.set_title("Original Test Image (32×32)")
ax.axis("off")
plt.tight_layout()
plt.show()
print(f"Image shape: {Y_image.shape}, dtype: {Y_image.dtype}")

## 4. Construct a Standard Cosine (DCT) Dictionary

The **Type-II Discrete Cosine Transform** matrix is a natural orthonormal
basis for real-valued signals and is widely used in image compression (JPEG).

For T-LARS we need one dictionary matrix **per tensor mode**.  Because our
image is 2-D (shape `(M, N)`), we build two DCT matrices: `D1` of shape
`(M, K1)` and `D2` of shape `(N, K2)`.

* **Complete** dictionary: `K = M` (square, orthonormal) — perfect recovery
  is always possible.
* **Overcomplete** dictionary: `K > M` — enables sparser representations
  but introduces redundancy.

Below we implement both options.

In [ ]:
def build_dct_dictionary(n: int, k: int | None = None) -> np.ndarray:
    """Build an (optionally overcomplete) DCT-II dictionary matrix.

    Parameters
    ----------
    n : int
        Signal length (number of rows).
    k : int or None, default=None
        Number of atoms (columns).  If ``None``, defaults to ``n``
        (complete, orthonormal basis).

    Returns
    -------
    D : np.ndarray, shape (n, k)
        Column-normalised DCT dictionary.

    Notes
    -----
    The DCT-II matrix element is defined as:

    .. math::

        D_{i,j} = \\cos\\!\\left(\\frac{\\pi\\,(2i + 1)\\,j}{2K}\\right),
        \\quad i = 0, \\dots, n{-}1,\\; j = 0, \\dots, K{-}1.

    Columns are then L2-normalised so that ``D.T @ D`` has unit diagonal.
    When ``K = n`` the dictionary is orthonormal.

    The DCT is a natural sparsifying basis for images because it
    concentrates energy into a few low-frequency coefficients — the
    same principle behind JPEG compression.

    Examples
    --------
    >>> D = build_dct_dictionary(32)
    >>> D.shape
    (32, 32)
    >>> np.allclose(D.T @ D, np.eye(32), atol=1e-12)
    True
    """
    if k is None:
        k = n
    i = np.arange(n).reshape(-1, 1)          # (n, 1)
    j = np.arange(k).reshape(1, -1)          # (1, k)
    D = np.cos(np.pi * (2 * i + 1) * j / (2 * k))
    # Column-normalise
    norms = np.linalg.norm(D, axis=0, keepdims=True)
    norms[norms == 0] = 1.0
    D = D / norms
    return D


# Build COMPLETE dictionaries (one per mode)
M, N = Y_image.shape
D1 = build_dct_dictionary(M)      # (32, 32) — row dictionary
D2 = build_dct_dictionary(N)      # (32, 32) — column dictionary

print(f"D1 shape: {D1.shape}  (row dictionary)")
print(f"D2 shape: {D2.shape}  (column dictionary)")
print(f"Total Kronecker columns: {D1.shape[1] * D2.shape[1]}")

# Visualise a few DCT atoms as tiny images
fig, axes = plt.subplots(1, 6, figsize=(12, 2))
for idx, ax in enumerate(axes):
    atom_2d = np.outer(D1[:, idx], D2[:, idx])   # rank-1 atom
    ax.imshow(atom_2d, cmap="RdBu_r", vmin=-0.15, vmax=0.15)
    ax.set_title(f"Atom {idx}", fontsize=9)
    ax.axis("off")
fig.suptitle("Sample DCT Atoms (rank-1 outer products)", fontsize=11)
plt.tight_layout()
plt.show()

## 5. Per-Iteration Reconstruction Helper

T-LARS does not expose a built-in callback, so we simulate per-iteration
snapshots by running the solver with **increasing iteration budgets** and
warm-starting from the previous solution via `coef_tensor`.

Each snapshot:
1. Prints the iteration number, active-set size, and residual norm.
2. Reconstructs the current image estimate.
3. Displays it inline so you can **watch the image emerge**.

In [ ]:
class IterativeReconstructor:
    """Run T-LARS at successive iteration checkpoints and display snapshots.

    Parameters
    ----------
    factor_matrices : list[np.ndarray]
        Dictionary matrices, one per tensor mode.
    Y : np.ndarray
        Target tensor (image).
    checkpoints : list[int]
        Iteration counts at which to capture a snapshot.
    tlars_kwargs : dict
        Extra keyword arguments forwarded to the TLARS constructor
        (e.g. ``tolerance``, ``l0_mode``, ``debug_mode``).

    Attributes
    ----------
    snapshots : list[dict]
        Captured snapshots, each containing keys ``'iter'``, ``'n_active'``,
        ``'residual_norm'``, and ``'image'``.

    Examples
    --------
    >>> rec = IterativeReconstructor([D1, D2], Y_image, [1, 5, 20])
    >>> rec.run()
    >>> len(rec.snapshots)
    3
    """

    def __init__(
        self,
        factor_matrices: List[np.ndarray],
        Y: np.ndarray,
        checkpoints: List[int],
        **tlars_kwargs,
    ) -> None:
        self.factor_matrices = factor_matrices
        self.Y = Y
        self.checkpoints = sorted(checkpoints)
        self.tlars_kwargs = tlars_kwargs
        self.snapshots: List[dict] = []

    def run(self, live_display: bool = True) -> List[dict]:
        """Execute T-LARS at every checkpoint and collect snapshots.

        Parameters
        ----------
        live_display : bool, default=True
            If ``True``, show each reconstruction inline (clears previous
            output for a "live-updating" effect in Jupyter).

        Returns
        -------
        snapshots : list[dict]
            One dict per checkpoint with keys ``'iter'``, ``'n_active'``,
            ``'residual_norm'``, ``'image'``.
        """
        self.snapshots.clear()

        for n_iter in self.checkpoints:
            # Run T-LARS with this iteration budget
            model = TLARS(
                iterations=n_iter,
                tolerance=1e-15,          # very tight — force it to use all iterations
                debug_mode=True,
                **self.tlars_kwargs,
            )
            model.fit(self.factor_matrices, self.Y)

            # Reconstruct
            Y_hat = model.predict(self.factor_matrices)
            residual = float(model.norm_r_[-1])
            n_active = len(model.active_columns_)

            snapshot = {
                "iter": model.n_iter_,
                "n_active": n_active,
                "residual_norm": residual,
                "image": np.array(Y_hat),
            }
            self.snapshots.append(snapshot)

            # Log to terminal / stdout
            print(
                f"[checkpoint] iter={model.n_iter_:>4d}  |  "
                f"active={n_active:>4d}  |  "
                f"||r||={residual:.6f}"
            )

            # Live display
            if live_display:
                ipy_display.clear_output(wait=True)
                fig, axes = plt.subplots(1, 2, figsize=(8, 4))
                axes[0].imshow(self.Y, cmap="gray", vmin=0, vmax=1)
                axes[0].set_title("Original")
                axes[0].axis("off")

                axes[1].imshow(Y_hat, cmap="gray", vmin=0, vmax=1)
                axes[1].set_title(
                    f"Iter {model.n_iter_} — {n_active} coeffs — ||r||={residual:.4f}"
                )
                axes[1].axis("off")
                plt.tight_layout()
                plt.show()

        return self.snapshots


print("IterativeReconstructor defined ✓")

## 6. Run T-LARS — Watch the Image Emerge

We run the solver at **14 checkpoints** from 1 to 500 iterations.
After the last checkpoint the final image will be displayed alongside the original.

> **Tip:** In a live Jupyter session the image updates in-place (the cell
> output clears and redraws at each checkpoint).  In a static render you will
> see only the final pair; re-run the cell to watch the animation.

In [ ]:
# Iteration checkpoints — increase density at early iterations where the
# most dramatic visual changes happen.
checkpoints = [1, 2, 3, 5, 8, 12, 20, 35, 50, 75, 100, 150, 250, 500]

reconstructor = IterativeReconstructor(
    factor_matrices=[D1, D2],
    Y=Y_image,
    checkpoints=checkpoints,
)

snapshots = reconstructor.run(live_display=True)

### Snapshot Gallery

A grid view of all captured checkpoints so you can see the progression at a glance.

In [ ]:
n_snaps = len(snapshots)
cols = min(n_snaps, 7)
rows = (n_snaps + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.2, rows * 2.5))
axes_flat = np.array(axes).flatten() if n_snaps > 1 else [axes]

for idx, (ax, snap) in enumerate(zip(axes_flat, snapshots)):
    ax.imshow(snap["image"], cmap="gray", vmin=0, vmax=1)
    ax.set_title(
        f"iter {snap['iter']}\n{snap['n_active']} coeffs",
        fontsize=8,
    )
    ax.axis("off")

# Hide unused axes
for ax in axes_flat[n_snaps:]:
    ax.axis("off")

fig.suptitle("T-LARS Reconstruction Progression", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 7. Analyse the Sparse Coefficient Path

After T-LARS completes we examine:
1. **Residual norm vs iteration** — how quickly the algorithm converges.
2. **Number of active coefficients** — the sparsity of the solution.
3. **Final reconstruction error** (MSE, PSNR).

In [ ]:
def compute_psnr(original: np.ndarray, reconstructed: np.ndarray, max_val: float = 1.0) -> float:
    """Compute Peak Signal-to-Noise Ratio (PSNR) between two images.

    Parameters
    ----------
    original : np.ndarray
        Ground-truth image.
    reconstructed : np.ndarray
        Reconstructed image (same shape as *original*).
    max_val : float, default=1.0
        Maximum possible pixel value.

    Returns
    -------
    psnr : float
        PSNR in decibels (dB).  Higher is better.

    Notes
    -----
    PSNR is defined as ``10 * log10(max_val**2 / MSE)``.
    A value above ~30 dB generally indicates good quality.
    """
    mse = np.mean((original - reconstructed) ** 2)
    if mse == 0:
        return float("inf")
    return float(10 * np.log10(max_val ** 2 / mse))


# ── Convergence plot ──────────────────────────────────────────────
iters   = [s["iter"] for s in snapshots]
resids  = [s["residual_norm"] for s in snapshots]
actives = [s["n_active"] for s in snapshots]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.semilogy(iters, resids, "o-", color="steelblue", linewidth=2)
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Residual Norm ||r||")
ax1.set_title("Convergence Curve")
ax1.grid(True, which="both", linewidth=0.3)

ax2.plot(iters, actives, "s-", color="darkorange", linewidth=2)
ax2.set_xlabel("Iteration")
ax2.set_ylabel("Active Coefficients")
ax2.set_title("Sparsity vs Iteration")
ax2.grid(True, linewidth=0.3)

plt.tight_layout()
plt.show()

# ── Summary table ─────────────────────────────────────────────────
final = snapshots[-1]
mse_val = float(np.mean((Y_image - final["image"]) ** 2))
psnr_val = compute_psnr(Y_image, final["image"])

print("┌──────────────────────────────────────────┐")
print(f"│ Final iteration         : {final['iter']:>10d}     │")
print(f"│ Active coefficients     : {final['n_active']:>10d}     │")
print(f"│ Total dictionary atoms  : {D1.shape[1] * D2.shape[1]:>10d}     │")
print(f"│ Sparsity ratio          : {final['n_active'] / (D1.shape[1]*D2.shape[1]):>10.2%}     │")
print(f"│ Residual norm           : {final['residual_norm']:>13.6f}  │")
print(f"│ MSE                     : {mse_val:>13.6e}  │")
print(f"│ PSNR                    : {psnr_val:>10.2f} dB   │")
print("└──────────────────────────────────────────┘")

## 8. Compare Final Reconstruction vs Original

In [ ]:
def compare_images(
    original: np.ndarray,
    reconstructed: np.ndarray,
    title: str = "Reconstruction Comparison",
) -> None:
    """Display original, reconstructed, and error images side-by-side.

    Parameters
    ----------
    original : np.ndarray
        Ground-truth image, values in [0, 1].
    reconstructed : np.ndarray
        Reconstructed image (same shape).
    title : str, default="Reconstruction Comparison"
        Figure super-title.

    Notes
    -----
    Three panels are shown:
    * **Left** – original image.
    * **Centre** – reconstructed image.
    * **Right** – absolute pixel-wise error (magnified for visibility).
    MSE, PSNR, and max-error are printed below the figure.
    """
    error = np.abs(original - reconstructed)
    mse = float(np.mean((original - reconstructed) ** 2))
    psnr = compute_psnr(original, reconstructed)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(original, cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(reconstructed, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("Reconstructed")
    axes[1].axis("off")

    im = axes[2].imshow(error, cmap="hot", vmin=0, vmax=error.max() or 1e-8)
    axes[2].set_title("Absolute Error")
    axes[2].axis("off")
    fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()

    print(f"  MSE      = {mse:.6e}")
    print(f"  PSNR     = {psnr:.2f} dB")
    print(f"  Max |err|= {error.max():.6e}")


compare_images(Y_image, snapshots[-1]["image"])

## 9. Edge Cases

### 9a — Noisy image
Add Gaussian noise and see how T-LARS naturally **denoises** via sparsity.

### 9b — Underdetermined system (compressed sensing)
Use an overcomplete dictionary simulating fewer measurements than unknowns.

### 9c — Degenerate signals
Test with an all-zero image and a constant image.

In [ ]:
# ─── 9a: Noisy Image ──────────────────────────────────────────────
rng = np.random.default_rng(42)
noise_level = 0.15
Y_noisy = np.clip(Y_image + rng.normal(0, noise_level, Y_image.shape), 0, 1)

# Run T-LARS on the noisy image (only 100 iterations — early stopping denoises)
model_noisy = TLARS(iterations=100, tolerance=noise_level, debug_mode=True)
model_noisy.fit([D1, D2], Y_noisy)
Y_noisy_hat = model_noisy.predict([D1, D2])

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(Y_image, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Original (clean)")
axes[0].axis("off")
axes[1].imshow(Y_noisy, cmap="gray", vmin=0, vmax=1)
axes[1].set_title(f"Noisy (σ={noise_level})")
axes[1].axis("off")
axes[2].imshow(Y_noisy_hat, cmap="gray", vmin=0, vmax=1)
axes[2].set_title(f"T-LARS denoised ({len(model_noisy.active_columns_)} coeffs)")
axes[2].axis("off")
plt.suptitle("Edge Case 9a: Noisy Image Recovery", fontsize=13)
plt.tight_layout()
plt.show()

psnr_noisy_input = compute_psnr(Y_image, Y_noisy)
psnr_noisy_recon = compute_psnr(Y_image, np.array(Y_noisy_hat))
print(f"  PSNR (noisy input vs clean)       : {psnr_noisy_input:.2f} dB")
print(f"  PSNR (T-LARS output vs clean)     : {psnr_noisy_recon:.2f} dB")
print(f"  Denoising improvement             : {psnr_noisy_recon - psnr_noisy_input:+.2f} dB")

In [ ]:
# ─── 9b: Overcomplete Dictionary (2× overcomplete) ──────────────
D1_oc = build_dct_dictionary(M, k=2 * M)   # (32, 64)
D2_oc = build_dct_dictionary(N, k=2 * N)   # (32, 64)
print(f"Overcomplete dictionaries: D1={D1_oc.shape}, D2={D2_oc.shape}")
print(f"Total atoms: {D1_oc.shape[1] * D2_oc.shape[1]}")

model_oc = TLARS(iterations=200, tolerance=0.01, debug_mode=True)
model_oc.fit([D1_oc, D2_oc], Y_image)
Y_oc_hat = model_oc.predict([D1_oc, D2_oc])

compare_images(Y_image, np.array(Y_oc_hat), title="Edge Case 9b: 2× Overcomplete DCT")
print(f"  Active coefficients: {len(model_oc.active_columns_)} / {D1_oc.shape[1]*D2_oc.shape[1]}")

In [ ]:
# ─── 9c: Degenerate Signals ──────────────────────────────────────
# All-zero image
Y_zero = np.zeros((M, N))
model_zero = TLARS(iterations=10, tolerance=0.01, debug_mode=True)
model_zero.fit([D1, D2], Y_zero)
print(f"All-zero image → active coefficients: {len(model_zero.active_columns_)}")
print(f"                 final ||r||: {model_zero.norm_r_[-1]:.6f}")

# Constant image
Y_const = np.full((M, N), 0.5)
model_const = TLARS(iterations=50, tolerance=0.001, debug_mode=True)
model_const.fit([D1, D2], Y_const)
Y_const_hat = model_const.predict([D1, D2])
print(f"\nConstant image → active coefficients: {len(model_const.active_columns_)}")
print(f"                 PSNR: {compute_psnr(Y_const, np.array(Y_const_hat)):.2f} dB")

## 10. Advanced Usage

### 10a — L0 vs L1 mode
T-LARS supports both greedy forward selection (`l0_mode=True`) and the full
LARS/LASSO path (`l0_mode=False`, the default).  L0 mode never removes a column
from the active set and tends to converge faster but may be slightly less accurate.

### 10b — Varying dictionary overcompleteness
Compare 1×, 2×, and 4× overcomplete DCT dictionaries.

### 10c — Parameter introspection
Demonstrate `get_params` / `set_params` (scikit-learn-style API).

In [ ]:
# ─── 10a: L0 vs L1 Mode ──────────────────────────────────────────
results = {}
for mode_name, l0 in [("L1 (LASSO path)", False), ("L0 (greedy)", True)]:
    model = TLARS(iterations=200, tolerance=0.01, l0_mode=l0, debug_mode=True)
    t0 = time.perf_counter()
    model.fit([D1, D2], Y_image)
    elapsed = time.perf_counter() - t0
    Y_hat = model.predict([D1, D2])
    psnr = compute_psnr(Y_image, np.array(Y_hat))
    results[mode_name] = {
        "n_iter": model.n_iter_,
        "n_active": len(model.active_columns_),
        "psnr": psnr,
        "time": elapsed,
    }
    print(f"{mode_name:>20s}  →  {model.n_iter_:>4d} iters, "
          f"{len(model.active_columns_):>3d} coeffs, "
          f"PSNR={psnr:.2f} dB, {elapsed:.3f}s")

# ── Bar chart comparison ─────────────────────────────────────────
labels = list(results.keys())
psnrs  = [results[k]["psnr"] for k in labels]
n_acts = [results[k]["n_active"] for k in labels]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
bars1 = ax1.bar(labels, psnrs, color=["steelblue", "darkorange"])
ax1.set_ylabel("PSNR (dB)")
ax1.set_title("Reconstruction Quality")
for bar, val in zip(bars1, psnrs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f"{val:.1f}", ha="center", fontsize=10)

bars2 = ax2.bar(labels, n_acts, color=["steelblue", "darkorange"])
ax2.set_ylabel("Active Coefficients")
ax2.set_title("Sparsity")
for bar, val in zip(bars2, n_acts):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             str(val), ha="center", fontsize=10)

plt.suptitle("L1 (LASSO) vs L0 (Greedy) Mode", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ─── 10b: Varying Dictionary Overcompleteness ────────────────────
overcompleteness_factors = [1, 2, 4]
oc_results = []

for factor in overcompleteness_factors:
    k = factor * M
    D1_f = build_dct_dictionary(M, k)
    D2_f = build_dct_dictionary(N, k)
    model = TLARS(iterations=200, tolerance=0.01, debug_mode=False)
    model.fit([D1_f, D2_f], Y_image)
    Y_hat = model.predict([D1_f, D2_f])
    psnr = compute_psnr(Y_image, np.array(Y_hat))
    total_atoms = D1_f.shape[1] * D2_f.shape[1]
    oc_results.append({
        "factor": factor,
        "k": k,
        "total_atoms": total_atoms,
        "n_active": len(model.active_columns_),
        "psnr": psnr,
        "image": np.array(Y_hat),
    })
    print(f"  {factor}× overcomplete  →  dict=({M},{k}), "
          f"atoms={total_atoms}, active={len(model.active_columns_)}, "
          f"PSNR={psnr:.2f} dB")

# Visual comparison
fig, axes = plt.subplots(1, len(oc_results) + 1, figsize=(4 * (len(oc_results) + 1), 4))
axes[0].imshow(Y_image, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Original")
axes[0].axis("off")
for idx, r in enumerate(oc_results):
    axes[idx + 1].imshow(r["image"], cmap="gray", vmin=0, vmax=1)
    axes[idx + 1].set_title(
        f"{r['factor']}× OC\n{r['n_active']}/{r['total_atoms']} coeffs\nPSNR={r['psnr']:.1f} dB",
        fontsize=9,
    )
    axes[idx + 1].axis("off")
fig.suptitle("Effect of Dictionary Overcompleteness", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ─── 10c: Parameter Introspection ─────────────────────────────────
model = TLARS(tolerance=0.05, l0_mode=True, iterations=500)

# get_params — returns all current parameters
print("get_params():")
for k, v in model.get_params().items():
    print(f"  {k:>22s} = {v!r}")

# set_params — update parameters (scikit-learn style)
model.set_params(tolerance=0.01, l0_mode=False)
print("\nAfter set_params(tolerance=0.01, l0_mode=False):")
print(f"  tolerance = {model.config.tolerance}")
print(f"  l0_mode   = {model.config.l0_mode}")